# 04 — Supervised ML: Pre-Treatment Completion Prediction & Leakage Prevention

## Objective
1. Engineer strictly pre-cutoff features (patient attributes at inquiry, provider historical baselines, country SLA baselines, and early communication logs within the first **72 hours**).
2. Exclude all post-treatment and target-leaking variables (`actual_revenue_inr`, `service_cost_inr`, `satisfaction_score`, stage/drop-off flags).
3. Train and compare **Logistic Regression** and **Random Forest** on an 80/20 stratified split.
4. Analyze classification thresholds (`0.30` to `0.60`) and feature importances.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data'

from src.data_quality import load_csvs
from src.modeling import (
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    FORBIDDEN_LEAKAGE_FEATURES,
    prepare_modeling_dataset,
    train_and_evaluate_models,
)

In [2]:
tables = load_csvs(DATA)
ml_df = prepare_modeling_dataset(
    tables['patient_journey'],
    tables['provider_master'],
    tables['communication_logs'],
    tables['country_reference'],
    cutoff_hours=72.0,
)
print('Numeric features (15):', NUMERIC_FEATURES)
print('Categorical features (7):', CATEGORICAL_FEATURES)
print('Excluded leakage columns:', FORBIDDEN_LEAKAGE_FEATURES)

Numeric features (15): ['intent_score', 'response_time_hours', 'engagement_score', 'estimated_case_value_inr', 'early_interactions', 'early_response_rate', 'early_avg_response_delay', 'early_followup_completed', 'avg_first_response_hours', 'historical_consultation_conversion_rate', 'historical_treatment_conversion_rate', 'follow_up_completion_rate', 'satisfaction_index', 'baseline_response_expectation_hours', 'market_priority_score']
Categorical features (7): ['country', 'age_group', 'gender', 'treatment_category', 'lead_source', 'budget_band', 'medical_urgency']
Excluded leakage columns: ['actual_revenue_inr', 'service_cost_inr', 'gross_margin_inr', 'satisfaction_score', 'treatment_date', 'length_of_stay_days', 'readmission_within_30d', 'follow_up_rating', 'current_stage', 'drop_off_flag', 'drop_off_stage', 'reason_for_drop_off']


## 1. Model Comparison & Threshold Analysis

In [3]:
results = train_and_evaluate_models(ml_df)
print('=== Test Set Model Performance (80/20 Stratified Split, n_test=300) ===')
display(results['metrics_df'])

print('=== Random Forest Operational Threshold Analysis ===')
display(results['thresholds_df'])

=== Test Set Model Performance (80/20 Stratified Split, n_test=300) ===
=== Random Forest Operational Threshold Analysis ===


,model,roc_auc,precision,recall,f1,tn,fp,fn,tp
0,Logistic Regression,0.819295,0.618705,0.774775,0.688000,136,53,25,86
1,Random Forest,0.869775,0.766990,0.711712,0.738318,165,24,32,79


,threshold,precision,recall,f1,tn,fp,fn,tp
0,0.3,0.602484,0.873874,0.713235,125,64,14,97
1,0.4,0.692308,0.810811,0.746888,149,40,21,90
2,0.5,0.766990,0.711712,0.738318,165,24,32,79
3,0.6,0.878378,0.585586,0.702703,180,9,46,65


## 2. Feature Importance & Interpretability

In [4]:
print('=== Top 15 Random Forest Features ===')
display(results['feature_importance_df'].head(15))

print('=== Top 15 Logistic Regression Standardized Coefficients & Odds Ratios ===')
display(results['logistic_coefficients_df'].head(15))

=== Top 15 Random Forest Features ===
=== Top 15 Logistic Regression Standardized Coefficients & Odds Ratios ===


,feature,importance,importance_pct
0,num__engagement_score,0.297029,29.702948
1,num__estimated_case_value_inr,0.130882,13.088229
2,num__response_time_hours,0.075531,7.553094
3,num__intent_score,0.073661,7.366077
4,num__early_avg_response_delay,0.047420,4.741979
5,num__follow_up_completion_rate,0.033156,3.315599
6,num__historical_consultation_conversion_rate,0.027728,2.772828
7,num__avg_first_response_hours,0.027080,2.708027
8,num__satisfaction_index,0.023968,2.396787
9,cat__treatment_category_Transplant Evaluation,0.022644,2.264394


,feature,coefficient,odds_ratio,abs_coefficient
0,num__engagement_score,1.208524,3.348539,1.208524
1,cat__lead_source_Email,0.377861,1.459160,0.377861
2,cat__lead_source_Referral Partner,-0.365729,0.693691,0.365729
3,num__response_time_hours,-0.297456,0.742705,0.297456
4,cat__treatment_category_Bariatric Surgery,0.296163,1.344690,0.296163
5,cat__medical_urgency_Urgent,-0.273801,0.760483,0.273801
6,cat__gender_Female,-0.270141,0.763272,0.270141
7,cat__budget_band_Low,-0.260008,0.771046,0.260008
8,cat__age_group_18-29,0.250443,1.284594,0.250443
9,cat__budget_band_High,0.213758,1.238323,0.213758
